# Auto-R diagnostic, arm 1/2 — `RandomRefs`: how many references does $f$ actually need?

$f(x) = E_{x' \sim p}[D^{2}_{IEM}(x,x')] / E_{x',x''\sim p}[D^{2}_{IEM}(x',x'')]$ is estimated from $R$ references.
With $R$ too small, $f$ is really *"how far is $x$ from these particular few dogs"*, and plenty of ordinary dogs
are far from a few specific ones — raising $\lambda$ then cannot help, because the reward is not measuring
novelty relative to $p$. This measures where that stops being true.

**The test.** `RefSelector.select(R=None)` sweeps a doubling ladder and accepts the smallest $R$ whose
**weighted-$\tau$** rank correlation $\tau(f_R, f_{2R}) \geq$ `tau_target` — i.e. where doubling the references stops
changing how $f$ *ranks* samples. $\tau$ is unitless, so the bar transfers across dimensions, but the achievable
**ceiling** may sit below the nominal 0.97 at $d = 65536$; `tau_history` exposes that ceiling directly.
The default weigher is hyperbolic in rank, so $\tau$ weights agreement **at the top of the ranking** — the novel
tail the tilt actually selects on — more than agreement in the bulk. That is the right emphasis here.

**Two arms, two failure modes.** This notebook runs `RandomRefs` (i.i.d. draws, uniform weights: *unbiased* for
$E_{x'\sim p}[D]$, variance $\propto 1/R$). `refset_auto_r_fps.ipynb` runs `WeightedFPSRefs` (greedy coverage in the
IEM metric, each reference weighted by the $p$-mass of its Voronoi cell: *consistent* rather than unbiased) as a
**separate Slurm job, in parallel**. Both import `auto_r_common.py`, so both see a bit-identical $f$ and rank the
**same** probe points — their $\tau$ curves are directly comparable, not merely comparable in distribution.

## Why the previous attempt failed, and what changed

Job 694256 died with CUDA OOM in *both* arms: `pairwise` fed `num_eps * rows` samples to the transformer in one
batch, so 64 probes meant 192 rows — 3.6 GB for a single activation. The notebook then "completed" with no
results, because `--allow-errors` swallowed it.

Two library changes since make this run possible, and they change the shape of the problem:

1. **`chunked_denoiser`** bounds the transformer batch, so the notebook no longer needs its own row-chunked
   `Distance` subclass. The OOM is fixed in the library, not worked around here.
2. **Reference score-bank caching** (`GlobalIEMDistance`, `cache_refs=True`) evaluates each point's score once per
   $(\gamma, W)$ instead of once per comparison. In the SMC this makes the per-sweep cost **independent of $R$**.

**It does not make $R$ free here.** This diagnostic's entire cost *is* building those banks, and that is linear in
the number of **distinct points scored**: $(N_\gamma - 1)\cdot N_\epsilon = 87$ transformer rows per point, plus 15 to
generate it, at a measured 0.22 s/row $\Rightarrow$ **~22 s per point**. So the budget is denominated in points
banked, not in $R$. What caching *did* buy is that a point scored once can be reused everywhere, which is what makes
the free extras below — and the FPS arm — possible at all.

## Settings, and what bounds them

| knob | value | reason |
|---|---|---|
| `N_GAMMA`, `NUM_EPS`, `GUIDANCE` | 30, 3, 1.5 | pinned to `flux_lambda_sweep_strong_2`, so the answer applies to *that* reward |
| `RAND_GRID` | (4, 8, 16, 32, 64, **128**) | smaller $R$ are **free** (column slices of one block), so the grid costs exactly what its top costs |
| `RAND_DRAWS` | 2 | 512 reference-points, ~3.2 h — the dominant cost, and what buys the independent-draw $\tau$ |
| `PROBE_SIZE` | 128 | $\tau$ resolution: sampling SE under independence is $\sqrt{2(2n+5)/(9n(n-1))}$ = 0.060 at 128, 0.086 at 64 |
| `TAU_TARGET` | 0.9 | with `fallback="best"`, so a lower ceiling **reports** instead of raising |

**Why the ladder stops at 128 (and how $R = 256$ is measured anyway).** A $\tau$ point *at* $R=256$ needs a 512-reference
block; that bank is 11.4 GB resident, and `score_bank`'s `torch.stack` needs it twice over at the peak — impossible
beside a 22.3 GB transformer on a 48 GB card. A **256**-reference block (5.4 GB) is the ceiling. But two
*independent* 256-draws concatenate into a genuine i.i.d. 512-set, so the last cells extract $\tau(f_{256}, f_{512})$
from blocks already computed, at **zero** additional cost.

**The free extras are the point of running 2 draws.** Both docstrings warn that the nested $R$-vs-$2R$ test is
**optimistic**: $f_R$ and $f_{2R}$ share $R$ reference terms. With two independent draws, $\tau(f_R^{A}, f_R^{B})$ —
same $R$, *disjoint* references — is the honest, unbiased version of the same question, at every $R$ up to 256.
Where the two curves separate *is* the nesting bias, measured rather than assumed.


In [ ]:
import os
import sys
import time

import matplotlib.pyplot as plt
import torch

T_START = time.time()

# The notebook is executed with cwd = its own directory (see auto_r_diagnostic.slurm), which is also
# where auto_r_common.py and the results files live.
OUT_DIR = os.path.abspath(os.getcwd())
REPO = os.path.abspath(os.path.join(OUT_DIR, "..", ".."))
for _p in (REPO, OUT_DIR):
    if _p not in sys.path:
        sys.path.insert(0, _p)

import auto_r_common as common      # the setup BOTH arms share -> identical f, identical probes

# Gated repo. Uses the HF token when this node mounts $HOME, and validates the cached 32 GB snapshot
# when it does not -- job 697403 died here with a 401 on a node without the mount.
common.check_model_access()

setup = common.build()
p_gen, D2 = setup.p_gen, setup.distance
probes = p_gen.sample(common.PROBE_SIZE, seed=common.SEED + 9973)   # memoized: the same tensor the
                                                                   # selector will request internally

# --- arm controls ---------------------------------------------------------------------------------
ARM = "random"
RAND_GRID = (4, 8, 16, 32, 64, 128)   # top drives the cost; the smaller entries are column slices
RAND_DRAWS = 2                        # 2 independent 256-ref draws -> the free extras in cell 4
MAX_REFS = 2 * max(RAND_GRID)         # 256 = the widest block, 5.4 GB of bank

# --- cost projection ------------------------------------------------------------------------------
# Exact, not estimated: every point is generated once (15 rows) and banked once (87 rows), and the
# point count is fully determined by the controls above. Verified by a CPU dry run at these grid
# sizes (65,520 rows), which counts the same builds this run will do.
POINTS = common.PROBE_SIZE + RAND_DRAWS * MAX_REFS
ROWS = POINTS * (common.GEN_ROWS + common.BANK_ROWS)
print(f"\n{POINTS} distinct points to bank "
      f"({common.PROBE_SIZE} probes + {RAND_DRAWS} x {MAX_REFS} refs)")
print(f"  {ROWS:,} transformer rows -> {common.hours(ROWS):.2f} h at {common.S_ROW} s/row "
      f"({ROWS * 0.25 / 3600:.2f} h at 0.25)")
print(f"  VRAM: probe bank {common.PROBE_SIZE * common.BANK_BYTES_PER_POINT / 2**30:.1f} GB + "
      f"{RAND_DRAWS} ref banks x {MAX_REFS * common.BANK_BYTES_PER_POINT / 2**30:.1f} GB "
      f"= {(common.PROBE_SIZE + RAND_DRAWS * MAX_REFS) * common.BANK_BYTES_PER_POINT / 2**30:.1f} GB "
      f"held at once")

# A 512-reference block -- the only way to get tau AT R=256 from a single draw -- is NOT runnable here:
_bank_512 = 512 * common.BANK_BYTES_PER_POINT / 2**30
print(f"\nfor contrast, a single 512-ref block would need {_bank_512:.1f} GB resident "
      f"({2 * _bank_512:.1f} GB at the stack peak in the library's score_bank) -> OOM. "
      f"Cell 4 gets R=256 from the two 256-draws instead, for free.")


In [ ]:
from creativity_measure.refset import RandomRefs

# =====================================================================================================
# CHECKPOINTING. `killable` is preemptible: job 698172 was preempted after 33 min and RESTARTED FROM
# ZERO on another node. The auto-R sweep is a single ~3 h library call, so with a preemption interval
# shorter than the run, the job never finishes -- it is not a slow run, it is a non-terminating one.
#
# Nothing below changes what is computed. It only decides where the expensive intermediates live:
#   probe score bank   2.9 GB, ~22 min   (rebuilt on every restart otherwise)
#   each draw's refs     67 MB, ~11 min   (handled by MemoSampler's disk cache)
#   the tau block, in COLUMN CHUNKS of 64 references, ~18 min each
#
# Column chunking is exact: block column r is pairwise(probes, ref_r), which depends on ref_r and the
# probes alone, so computing the columns in groups and concatenating gives the same matrix as one
# call. What it buys is that a preemption can destroy at most one chunk (~18 min) instead of the
# whole 1.6 h draw -- below the observed preemption interval, which is the property that matters.
# =====================================================================================================
CKPT_DIR = os.path.join(OUT_DIR, "ckpt_random")
os.makedirs(CKPT_DIR, exist_ok=True)
COL_CHUNK = 64

p_gen.cache_dir = CKPT_DIR          # sample() now persists; probes/refs survive a requeue
PROBE_BANK_CKPT = os.path.join(CKPT_DIR, "probe_bank.pt")
if D2.load_bank(probes, PROBE_BANK_CKPT):
    print(f"[ckpt] probe score bank restored from disk ({common.PROBE_SIZE} points, "
          f"~{common.PROBE_SIZE * common.BANK_ROWS * common.S_ROW / 60:.0f} min saved)")


class CheckpointedRandomRefs(RandomRefs):
    """`RandomRefs` whose tau blocks are computed in column chunks, cached on disk, and kept in RAM.

    Two jobs in one override:

    * RESUMABILITY (see the header): each 64-reference chunk is saved as soon as it lands, so a
      requeued job re-does at most the chunk it was in the middle of.
    * ANALYSIS. The library discards the blocks after slicing them for the tau sweep. They are ~2.6 h
      of transformer time and contain strictly more than the nested tau the rule reports, so cell 4
      mines them for the independent-draw tau and for a tau point at R=256. No IEM evaluation is
      repeated and the auto-R result is bit-identical either way.
    """

    def __init__(self, *args, **kwargs) -> None:
        super().__init__(*args, **kwargs)
        self.blocks: list[torch.Tensor] = []

    def _tau_block(self, probes_arg, max_r, draw):
        assert self.distance is not None
        n_refs = 2 * max_r
        done = os.path.join(CKPT_DIR, f"block_draw{draw}.pt")
        if os.path.exists(done):
            block = torch.load(done, map_location="cpu", weights_only=False)
            if tuple(block.shape) == (probes_arg.shape[0], n_refs):
                print(f"[ckpt] draw {draw}: complete block restored from disk "
                      f"({block.shape[0]}x{block.shape[1]})")
                self.blocks.append(block.float())
                return block.to(probes_arg.device, probes_arg.dtype)
            print(f"[ckpt] block_draw{draw}.pt has the wrong shape {tuple(block.shape)}; recomputing")

        refs = self._refs_for_size(n_refs, draw=draw)      # disk-cached by the sampler
        cols: list[torch.Tensor] = []
        for start in range(0, n_refs, COL_CHUNK):
            stop = min(start + COL_CHUNK, n_refs)
            part_path = os.path.join(CKPT_DIR, f"block_draw{draw}_cols{start}_{stop}.pt")
            if os.path.exists(part_path):
                part = torch.load(part_path, map_location="cpu", weights_only=False)
                print(f"[ckpt] draw {draw}: columns {start}:{stop} restored from disk")
            else:
                t_chunk = time.time()
                part = self.distance.pairwise(probes_arg, refs[start:stop]).detach().float().cpu()
                common.atomic_save(part, part_path)
                print(f"[ckpt] draw {draw}: columns {start}:{stop} computed in "
                      f"{(time.time() - t_chunk) / 60:.1f} min and saved   |   {D2.stats()}")
                # The probe bank exists after the first chunk; persist it once so a restart skips it.
                if not os.path.exists(PROBE_BANK_CKPT):
                    D2.save_bank(probes_arg, PROBE_BANK_CKPT)
            cols.append(part)

        block = torch.cat(cols, dim=1)                     # (K, 2*max_r)
        common.atomic_save(block, done)
        for start in range(0, n_refs, COL_CHUNK):          # the chunks are now redundant
            part_path = os.path.join(CKPT_DIR,
                                     f"block_draw{draw}_cols{start}_{min(start + COL_CHUNK, n_refs)}.pt")
            if os.path.exists(part_path):
                os.remove(part_path)
        self.blocks.append(block)
        return block.to(probes_arg.device, probes_arg.dtype)


sel_rand = CheckpointedRandomRefs(
    p_gen, distance=D2, seed=common.SEED,
    auto_r_grid=RAND_GRID, tau_target=common.TAU_TARGET, probe_size=common.PROBE_SIZE,
    auto_r_draws=RAND_DRAWS, fallback="best",
)

t0 = time.time()
R_rand = sel_rand.select(None).shape[0]
dt_rand = time.time() - t0

rows_done = D2.rows_scored + p_gen.rows_generated
print(f"\nRandomRefs auto-R -> R = {R_rand}   ({dt_rand / 60:.1f} min, projected "
      f"{common.hours(ROWS) * 60:.0f} min)")
print(f"realized {dt_rand / max(rows_done, 1):.3f} s/row over {rows_done:,} rows "
      f"(assumed {common.S_ROW}; after a restart this is meaningless -- {p_gen.disk_hits} "
      f"sample checkpoints were reused)")
print(D2.stats())

print(f"\nNESTED tau -- the library's own rule (f_R and f_2R share R references, so optimistic):")
print(f"{'R':>5} {'tau(f_R, f_2R)':>15}  {'>= target':>9}")
for R, tau in sel_rand.tau_history:
    print(f"{R:5d} {tau:15.4f}  {'yes' if tau >= common.TAU_TARGET else 'no':>9}")
ceiling_rand = max((t for _, t in sel_rand.tau_history), default=float("nan"))
print(f"\nobserved tau ceiling: {ceiling_rand:.4f}   (target {common.TAU_TARGET})")
tau_at_64 = dict(sel_rand.tau_history).get(64)
if tau_at_64 is not None:
    print(f"tau at R=64 (the strong-tilt sweeps' choice): {tau_at_64:.4f} -> "
      + ("STABLE: doubling to 128 leaves the ranking alone" if tau_at_64 >= common.TAU_TARGET
         else "UNSTABLE: doubling to 128 reorders f, so R=64 is still distorting the reward"))

payload = {
    "config": common.config_dict(),
    "arm": ARM,
    "controls": {"grid": list(RAND_GRID), "draws": RAND_DRAWS, "max_refs": MAX_REFS,
                 "col_chunk": COL_CHUNK},
    "nested_tau": sel_rand.tau_history,
    "auto_R": R_rand,
    "ceiling": ceiling_rand,
    "minutes": dt_rand / 60,
    "rows": rows_done,
    "s_per_row": dt_rand / max(rows_done, 1),
}
common.save_results(OUT_DIR, ARM, payload)      # written now: the extras below are free but not free
                                                # of risk, and this result must survive them


In [ ]:
# --- the free extras: what two independent draws give that the nested rule cannot ------------------
# No IEM evaluation happens in this cell. Everything below is arithmetic on the two (128 x 256) blocks
# stashed above, which is why the 2-draw ladder is worth its 3.2 h.
#
# 1. INDEPENDENT-DRAW tau(f_R^A, f_R^B): same R, DISJOINT reference sets. The nested test compares f_R
#    with f_2R, which contains it -- the shared R terms inflate tau, and both selectors' docstrings
#    say so without quantifying it. Here the two estimates share nothing, so the agreement is real.
#    The gap between the two curves IS the nesting bias, measured.
# 2. tau AT R=256, from concatenating the draws: two independent 256-sets ARE one i.i.d. 512-set, and
#    draw A is nested in it exactly as the library's rule requires -> tau(f_256, f_512) with no
#    512-reference bank ever built (which would not fit in VRAM).
# 3. The RELATIVE STANDARD ERROR of f itself. tau answers "is the ranking stable"; the tilt also needs
#    the SCALE to be stable, because lam is set in units of lam_s = 1 / std_p(f). If an independent
#    reference draw moves a sample's f by a fraction rho of std_p(f), then a tilt at m * lam_s is
#    really a tilt at (m +/- m*rho) * lam_s -- reference noise masquerading as a lambda error.
#    sd(f_A - f_B) = sqrt(2) * SE, so SE = sd(f_A - f_B) / sqrt(2).
from scipy.stats import weightedtau

blocks = sel_rand.blocks
assert len(blocks) == RAND_DRAWS and blocks[0].shape[1] == MAX_REFS, "unexpected block shapes"
bA, bB = blocks[0], blocks[1]
bAB = torch.cat([bA, bB], dim=1)                    # (K, 512): a genuine i.i.d. 512-reference set

EXT_GRID = [R for R in (4, 8, 16, 32, 64, 128, 256) if R <= MAX_REFS]


def f_of(block: torch.Tensor, R: int) -> torch.Tensor:
    """RandomRefs' own reduction: the uniform mean over the first R reference columns."""
    return block[:, :R].mean(dim=1)


def tau_of(u: torch.Tensor, v: torch.Tensor) -> float:
    return float(weightedtau(u.numpy(), v.numpy())[0])


indep_tau, rel_se, nested_1draw = [], [], []
for R in EXT_GRID:
    fA, fB = f_of(bA, R), f_of(bB, R)
    indep_tau.append((R, tau_of(fA, fB)))
    pooled_sd = float(torch.cat([fA, fB]).std())            # spread of f ACROSS probes ~ std_p(f)
    rel_se.append((R, float((fA - fB).std()) / (2 ** 0.5) / pooled_sd))
    if 2 * R <= MAX_REFS:                                   # single-draw nested tau, for calibration
        nested_1draw.append((R, tau_of(fA, f_of(bA, 2 * R))))

tau_256_nested = tau_of(f_of(bA, MAX_REFS), bAB.mean(dim=1))
nested_1draw.append((MAX_REFS, tau_256_nested))

print("INDEPENDENT-draw tau (same R, disjoint references) vs the NESTED tau at the same R:")
print(f"{'R':>5} {'tau_indep':>10} {'tau_nested':>11} {'bias':>7} {'SE(f)/std_p(f)':>15}")
_nested = dict(nested_1draw)
for (R, ti), (_, rse) in zip(indep_tau, rel_se):
    tn = _nested.get(R)
    bias = "" if tn is None else f"{tn - ti:+.3f}"
    tn_s = "" if tn is None else f"{tn:.4f}"
    print(f"{R:5d} {ti:10.4f} {tn_s:>11} {bias:>7} {rse:15.3f}")

print(f"\ntau(f_256, f_512) = {tau_256_nested:.4f}   [nested, from the two draws concatenated -- the "
      f"R=256 answer, at no extra cost]")
se = (2 * (2 * common.PROBE_SIZE + 5) / (9 * common.PROBE_SIZE * (common.PROBE_SIZE - 1))) ** 0.5
print(f"tau sampling SE at {common.PROBE_SIZE} probes ~ {se:.3f} -> read the CURVE; treat gaps "
      f"below ~{se:.2f} as ties.")

# What the relative SE means for the tilt: the sweeps run m = 3 (lam = 3 * lam_s).
M_TILT = 3.0
print(f"\nreference noise expressed as a lambda error at m = {M_TILT:g} "
      f"(the strong-tilt sweeps' setting):")
for R, rse in rel_se:
    print(f"  R={R:4d}  an independent reference draw shifts f by {rse:.3f} std_p(f) "
          f"= the effect of m moving by {M_TILT * rse:.2f}")

payload["independent_tau"] = indep_tau
payload["nested_tau_single_draw"] = nested_1draw
payload["rel_se"] = rel_se
payload["tau_256_nested"] = tau_256_nested
payload["tau_probe_se"] = se
common.save_results(OUT_DIR, ARM, payload)


In [ ]:
# --- figure + verdict -------------------------------------------------------------------------------
fig, (ax, ax2) = plt.subplots(1, 2, figsize=(11.5, 4.3))

ax.plot([R for R, _ in sel_rand.tau_history], [t for _, t in sel_rand.tau_history], "o-",
        color="tab:blue", label=f"nested, draw-averaged (library rule) -> R = {R_rand}")
ax.plot([R for R, _ in nested_1draw], [t for _, t in nested_1draw], "s--", color="tab:cyan",
        lw=1, ms=4, label="nested, single draw (incl. R = 256)")
ax.plot([R for R, _ in indep_tau], [t for _, t in indep_tau], "o-", color="tab:red",
        label="INDEPENDENT draws (unbiased)")
ax.axhline(common.TAU_TARGET, color="crimson", ls="--", lw=1,
           label=f"tau_target = {common.TAU_TARGET}")
ax.axvline(64, color="grey", ls=":", lw=1, label="R = 64 (strong-tilt sweeps)")
ax.fill_between([min(EXT_GRID), max(EXT_GRID)], 1 - se, 1, color="grey", alpha=0.12,
                label=f"within tau SE ({se:.3f}) of 1")
ax.set_xscale("log", base=2)
ax.set_xticks(EXT_GRID); ax.set_xticklabels([str(r) for r in EXT_GRID])
ax.set_xlabel("R"); ax.set_ylabel(r"weighted-$\tau$")
ax.set_title("Rank stability of $f$ vs reference count")
ax.grid(alpha=0.3); ax.legend(fontsize=7.5, loc="lower right")

ax2.plot([R for R, _ in rel_se], [v for _, v in rel_se], "o-", color="tab:purple")
ax2.set_xscale("log", base=2); ax2.set_yscale("log")
ax2.set_xticks(EXT_GRID); ax2.set_xticklabels([str(r) for r in EXT_GRID])
ax2.set_xlabel("R"); ax2.set_ylabel(r"SE($f$) / std$_p(f)$")
ax2.set_title(r"Scale stability: reference noise in units of $\lambda_s^{-1}$")
ax2.axhline(1 / M_TILT / 10, color="grey", ls=":", lw=1)
ax2.text(EXT_GRID[0], 1 / M_TILT / 10, r"  10% of a unit step in $m$", fontsize=7.5,
         va="bottom", color="dimgrey")
ax2.grid(alpha=0.3, which="both")

fig.suptitle(f"RandomRefs auto-R   (d={common.LATENT_DIM}, N_GAMMA={common.N_GAMMA}, "
             f"NUM_EPS={common.NUM_EPS}, {common.PROBE_SIZE} probes, {RAND_DRAWS} draws)",
             fontsize=10)
plt.tight_layout()
fig.savefig(os.path.join(OUT_DIR, "auto_r_tau_random.png"), dpi=140, bbox_inches="tight")
plt.show()

# --- verdict ------------------------------------------------------------------------------------
print("=" * 92)
ind = dict(indep_tau)
for R in (4, 64, 128, 256):
    if R in ind:
        print(f"R={R:4d}: independent-draw tau = {ind[R]:.4f}"
              + ("   <- the strong-tilt sweeps' R" if R == 64 else ""))
best_indep = max((t for _, t in indep_tau), default=float("nan"))
adequate = [R for R, t in indep_tau if t >= common.TAU_TARGET]
print(f"\nsmallest R clearing tau_target={common.TAU_TARGET} on the UNBIASED test: "
      f"{adequate[0] if adequate else 'none (ceiling %.4f)' % best_indep}")
print(f"the library's nested rule chose R = {R_rand}; the gap between the curves is the nesting bias.")
_per_ref = (common.GEN_ROWS + common.BANK_ROWS) * common.S_ROW
print("\nCost of acting on this: the SMC's per-sweep cost is independent of R (the reference bank is")
print(f"built once and cached), so raising R costs a ONE-TIME {_per_ref:.0f} s per reference at setup "
      f"({256 * _per_ref / 3600:.1f} h at R=256),")
print(f"plus {common.BANK_BYTES_PER_POINT / 2**20:.0f} MB of resident VRAM per reference "
      f"(R=256 -> {256 * common.BANK_BYTES_PER_POINT / 2**30:.1f} GB, which still fits beside FLUX).")
print("R is therefore no longer a budget question for sampling -- only for THIS diagnostic.")
print("=" * 92)
print(f"\ntotal notebook time: {(time.time() - T_START) / 60:.1f} min   |   {D2.stats()}")


In [ ]:
# --- cross-arm figure, if the FPS job has already landed --------------------------------------------
# The two arms run as separate Slurm jobs, so this is opportunistic: whichever finishes second draws
# the joint plot. `compare_arms.py` does the same thing standalone (no GPU) once both JSONs exist.
import json

_fps_path = common.results_path(OUT_DIR, "fps")
if os.path.exists(_fps_path):
    import compare_arms

    compare_arms.plot(OUT_DIR)
else:
    print(f"{os.path.basename(_fps_path)} not present yet -- the WeightedFPS job is still running or "
          f"has not been submitted.\nRun `python compare_arms.py` in this directory once it lands "
          f"(CPU only, seconds).")


## Takeaways

*(Job 698393, n-805, 3.1 h. 65,340 rows at a realized **0.170 s/row** on an L40S — the 0.22 assumption,
taken from an A6000, is ~30% conservative. 0 cache evictions against the 14 GB budget.)*

### 1. The library's auto-R rule is not usable at $d = 65536$ — it stopped at $R = 4$

`select(None)` returned **R = 4** on nested $\tau = 0.918 \geq 0.9$. But two *independent* 4-reference sets
rank the same 128 probes at $\tau = 0.696$. The nesting bias is **+0.211 at $R=4$** and $\leq 0.043$ at every
larger $R$ — i.e. it is largest exactly where the rule stops, which is the worst possible place for it.

| $R$ | 4 | 8 | 16 | 32 | 64 | 128 | 256 |
|---|---|---|---|---|---|---|---|
| **independent-draw $\tau$** (unbiased) | 0.696 | 0.903 | 0.932 | 0.953 | **0.960** | 0.970 | 0.991 |
| nested $\tau$ (the library's rule) | 0.918 | 0.931 | 0.975 | 0.966 | 0.987 | 0.974 | 0.995 |
| SE$(f)$ / std$_p(f)$ | 0.530 | 0.223 | 0.148 | 0.108 | **0.087** | 0.061 | 0.031 |

**Consequence for the repo:** never take `RefSelector.select(None)`'s $R$ at face value at this scale. Either
read it against an independent-draw curve, or raise `tau_target` to ~0.99 to absorb a bias that is only large
at small $R$. This applies to *both* selectors — the bias is in the shared rule, not in `RandomRefs`.

### 2. $R = 64$ was **not** what was wrong with the strong-tilt sweeps

Independent $\tau = 0.960$ at $R = 64$ — above the bar and within 2 $\tau$-SE (0.060) of the ceiling. The
sweeps' reward ranked samples stably. **So "the tilt moved $f$ by ≈2σ yet produced ordinary dogs" is not
explained by too few references**, which was this notebook's original hypothesis. Look at `GUIDANCE`
(it sets how wide $p$ is — free), at `N_GAMMA`, or at the sampler.

### 3. The live problem is **scale** stability, not rank stability — and it has never been accounted for

$\lambda$ is calibrated in units of $\lambda_s = 1/\mathrm{std}_p(f)$, so an error in $f$'s *scale* is an error in
$\lambda$. At $R = 64$ an independent reference draw shifts $f$ by $0.087\,\mathrm{std}_p(f)$ = **±0.26 in
effective $m$** — a quarter of a unit step on a grid whose steps are $m \in \{1,2,3\}$. Part of the
run-to-run variation in the $\lambda$ sweeps is therefore the reference draw, not $\lambda$.

At $R = 256$ that falls to 0.031 (**±0.09 in $m$**), i.e. an order of magnitude below a unit step.

### 4. Recommendation: $R = 256$ for the next sweep (or 128 as the cheap compromise)

Now affordable, because the per-sweep SMC cost is **independent of $R$** — the price is a one-time bank build
and resident VRAM:

| $R$ | one-time bank build | resident VRAM | ±$m$ from reference noise |
|---|---|---|---|
| 64 (current) | 0.4 h | 1.4 GB | ±0.26 |
| 128 | 0.8 h | 2.9 GB | ±0.18 |
| **256** | **1.6 h** | **5.4 GB** | **±0.09** |

5.4 GB still fits beside the 22.3 GB transformer on a 48 GB card. Do **not** go to 512 in one block: the bank
is 11.4 GB and the library's `score_bank` needs it twice over at its `torch.stack` peak.

### 5. `NUM_EPS = 3` is not the limiting factor

Independent $\tau$ reaches 0.9905 at $R = 256$, so Brownian-path noise at `NUM_EPS=3` does not prevent a stable
ranking. The $\tau$ ceiling worry in CLAUDE.md does not bind at this configuration.

**Caveats.** $\tau$-SE at 128 probes is 0.060, so treat gaps below that as ties — the $R \geq 32$ portion of the
independent curve is one such plateau. The independent curve is itself two draws, so it is one realisation of a
noisy quantity. `N_GAMMA`/`NUM_EPS`/`GUIDANCE` are pinned to `flux_lambda_sweep_strong_2`, so all of this
describes *that* reward.
